In [1]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders.csv_loader import CSVLoader

from langchain.text_splitter import RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')

# CSV 데이터 로드
loader = CSVLoader(file_path='finetune_data.csv', source_column='Title', encoding='utf-8-sig')
wiki_data = loader.load()

# Embedding Model 로드 (Local 다운로드 방식)
embeddings_model = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sroberta-nli',
#     model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True},
)

# 데이터 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=6000, chunk_overlap=200)
split_wiki_data = text_splitter.split_documents(wiki_data)

# Path 설정
DB_PATH_HF = "./ChromaDB_Wiki"

# ChromaDB 설정
vectorstore = Chroma.from_documents(
    documents=split_wiki_data,
    embedding=embeddings_model,
    persist_directory = DB_PATH_HF
)

def return_wiki_data(query_string: str, return_counts:int) -> dict:
    docs = vectorstore.similarity_search(query_string, k=return_counts)
    return_form = []
    for idx, content in enumerate(docs):
        create_dict = {
        'SimilarityRank':idx + 1,
        'Title':content.page_content[7 : content.page_content.find('Contents')],
        'Content':content.page_content[content.page_content.find('Contents') + 10 : ]
        }
        return_form.append(create_dict.copy())
    
    return return_form

The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


In [18]:
for i in return_wiki_data("아나콘다 가상환경은 어떻게 구축해", 3):
    pprint.pprint(i)

{'Content': '# ---- 아나콘다 가상환경 구축(인터넷 가능 환경에서)\n'
            '# 가상환경 py3.7 만들고 패키지 설치\n'
            '# (base) 는 Anaconda prompt 환경\n'
            '(base) conda create --name py3.7 python=3.7\n'
            '(base) conda install --name py3.7 pymssql\n'
            '# conda install --name py3.7 cx_Oracle\n'
            '(base) conda install --name py3.7 numpy\n'
            '(base) conda install --name py3.7 pandas\n'
            '(base) conda install --name py3.7 matplotlib\n'
            '(base) conda install --name py3.7 openpyxl\n'
            '(base) conda install --name py3.7 jupyter\n'
            '#conda install --name py3.7 fastapi\n'
            '(base) conda install --name py3.7 scikit-learn\n'
            '#conda install --name py3.7 uvicorn\n'
            '#아래 pip install 을 가상환경에서 된다고 하는 말이 있으나 잘 안됨.ㅡㅡ\n'
            '#가상환경안에서 pip install fastapi\n'
            '#가상환경안에서 pip install matplotlib\n'
            '#가상환경안에서 pip install seaborn\n'
            '# ---- 아나콘다 가상환경을 오프